# 09 CatBoost Tuning
Use CatBoost native categorical handling; do not one-hot encode its categorical inputs.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import joblib
from sklearn.metrics import average_precision_score, roc_auc_score

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.config import MODEL_DIR, PROCESSED_DATA_DIR
from src.catboost_model import tuned_catboost
from src.preprocessing import prepare_catboost_frame

X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").squeeze("columns")
X_validation = pd.read_csv(PROCESSED_DATA_DIR / "X_validation.csv")
y_validation = pd.read_csv(PROCESSED_DATA_DIR / "y_validation.csv").squeeze("columns")
X_train, cat_indices = prepare_catboost_frame(X_train)
X_validation, _ = prepare_catboost_frame(X_validation, X_train.columns[X_train.dtypes == "object"].tolist())
catboost = tuned_catboost(iterations=250, depth=5, learning_rate=0.05)
catboost.fit(X_train, y_train, cat_features=cat_indices, eval_set=(X_validation, y_validation), early_stopping_rounds=30, verbose=False)
validation_probabilities = catboost.predict_proba(X_validation)[:, 1]
print("CatBoost validation ROC-AUC:", roc_auc_score(y_validation, validation_probabilities))
print("CatBoost validation PR-AUC:", average_precision_score(y_validation, validation_probabilities))
MODEL_DIR.joinpath("tuned_catboost").mkdir(parents=True, exist_ok=True)
joblib.dump(catboost, MODEL_DIR / "tuned_catboost" / "notebook_model.joblib")